# DreamBooth LoRA Studio — Google Colab

Personalized image & video generation with DreamBooth + LoRA on SDXL/FLUX.

**GPU Requirements:**
- **T4 (free tier):** SDXL training & inference, AnimateDiff
- **A100/V100 (paid):** FLUX.1-dev training, CogVideoX

**Runtime → Change runtime type → GPU** before running.

## 0 — Setup & Environment

In [10]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4
VRAM: 15.6 GB
PyTorch: 2.10.0+cu128
CUDA: 12.8


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Persistent storage on Google Drive (survives runtime restarts)
import os

PROJECT_ROOT = "/content/ImgGen"
DRIVE_ROOT = "/content/drive/MyDrive/ImgGen"

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Symlink heavy directories to Google Drive so models persist
for folder in ["models", "lora_weights", "data"]:
    drive_path = os.path.join(DRIVE_ROOT, folder)
    local_path = os.path.join(PROJECT_ROOT, folder)
    os.makedirs(drive_path, exist_ok=True)
    if not os.path.exists(local_path):
        os.symlink(drive_path, local_path)
        print(f"  {folder}/ → Google Drive")

# outputs stay local (faster I/O)
os.makedirs(os.path.join(PROJECT_ROOT, "outputs"), exist_ok=True)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Drive root:   {DRIVE_ROOT}")


Project root: /content/ImgGen
Drive root:   /content/drive/MyDrive/ImgGen


In [13]:
import subprocess, sys

# Install numpy first to avoid ABI mismatch with Colab's preinstalled packages
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.26,<2"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.30.0",
    "transformers==4.44.0",
    "accelerate==0.33.0",
    "peft==0.12.0",
    "bitsandbytes==0.43.3",
    "datasets==2.20.0",
    "huggingface_hub==0.24.5",
    "controlnet_aux==0.0.9",
    "xformers",
    "einops",
    "tomesd",
    "imageio[ffmpeg]",
    "opencv-python-headless",
    "gradio==4.42.0",
    "safetensors",
    "sentencepiece",
    "protobuf",
])

print("\nAll dependencies installed. Restarting runtime now...")
print("After restart: run all cells BELOW this one (skip this cell).")

import os
os.kill(os.getpid(), 9)

In [14]:
# Log in to Hugging Face (needed for FLUX.1-dev gated model)
from huggingface_hub import notebook_login
notebook_login()

In [15]:
# Write the central config — all paths point to our Colab layout

os.makedirs(os.path.join(PROJECT_ROOT, "configs"), exist_ok=True)

config_code = '''
from pathlib import Path

ROOT = Path("/content/ImgGen")

DATA_DIR = ROOT / "data"
SUBJECT_DIR = DATA_DIR / "subject_images"
REG_DIR = DATA_DIR / "reg_images"
STYLE_DIR = DATA_DIR / "style_images"
CONTROLNET_DIR = DATA_DIR / "controlnet_pairs"
VIDEO_DIR = DATA_DIR / "video_clips"
PROCESSED_DIR = DATA_DIR / "processed"

MODEL_DIR = ROOT / "models"
FLUX_PATH = MODEL_DIR / "flux1-dev"
SDXL_PATH = MODEL_DIR / "sdxl-base"
SDXL_VAE_PATH = MODEL_DIR / "sdxl-vae-fix"
COGVIDEO_PATH = MODEL_DIR / "cogvideox-5b"

LORA_DIR = ROOT / "lora_weights"
SUBJECT_LORA_PATH = LORA_DIR / "subject_lora"
STYLE_LORA_PATH = LORA_DIR / "style_lora"

OUTPUT_DIR = ROOT / "outputs"

TRIGGER_TOKEN = "sks person"
STYLE_TRIGGER = "in the style of artx"

SUBJECT_LORA_RANK = 64
SUBJECT_LORA_ALPHA = 64
SUBJECT_LR = 1e-4
SUBJECT_STEPS = 1000
SUBJECT_BATCH_SIZE = 1
SUBJECT_GRAD_ACCUM = 4

STYLE_LORA_RANK = 32
STYLE_LORA_ALPHA = 32
STYLE_LR = 5e-5
STYLE_STEPS = 500

DEFAULT_STEPS = 28
DEFAULT_GUIDANCE = 3.5
DEFAULT_SEED = 42

DTYPE = "bfloat16"
ENABLE_CPU_OFFLOAD = True
ENABLE_SEQUENTIAL_OFFLOAD = False
'''.strip()

with open(os.path.join(PROJECT_ROOT, "configs", "default.py"), "w") as f:
    f.write(config_code)

with open(os.path.join(PROJECT_ROOT, "configs", "__init__.py"), "w") as f:
    f.write("from configs.default import *\n")

print("Config written.")

Config written.


In [16]:
import sys
sys.path.insert(0, PROJECT_ROOT)

from configs.default import *
print(f"ROOT:      {ROOT}")
print(f"DATA_DIR:  {DATA_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")
print(f"LORA_DIR:  {LORA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

ROOT:      /content/ImgGen
DATA_DIR:  /content/ImgGen/data
MODEL_DIR: /content/ImgGen/models
LORA_DIR:  /content/ImgGen/lora_weights
OUTPUT_DIR: /content/ImgGen/outputs


## Phase 1 — Download Datasets

In [19]:
from datasets import load_dataset, load_from_disk, Dataset, Features, Value, Image as HFImage
from huggingface_hub import snapshot_download

# 1/5 Subject images
if not (SUBJECT_DIR.exists() and any(SUBJECT_DIR.iterdir())):
    print("[1/5] Downloading subject dataset (diffusers/dog-example)...")
    ds = load_dataset("diffusers/dog-example", split="train")
    ds.save_to_disk(str(SUBJECT_DIR))
    print(f"  Saved {len(ds)} samples")
else:
    print("[skip] Subject dataset already exists")

# 2/5 Regularisation faces
if not (REG_DIR.exists() and any(REG_DIR.iterdir())):
    print("[2/5] Downloading regularisation faces...")
    ds = load_dataset("multimodalart/faces-prior-preservation", split="train")
    ds.save_to_disk(str(REG_DIR))
    print(f"  Saved {len(ds)} samples")
else:
    print("[skip] Regularisation dataset already exists")

# 3/5 Style (WikiArt first 5000)
if not (STYLE_DIR.exists() and any(STYLE_DIR.iterdir())):
    print("[3/5] Downloading style dataset (WikiArt, first 5000)...")
    stream = load_dataset("Artificio/WikiArt", split="train", streaming=True)
    subset = list(stream.take(5000))
    features = Features({"image": HFImage(), "text": Value("string")})
    rows = [{"image": s["image"], "text": s.get("artist", "unknown")} for s in subset if s.get("image")]
    ds = Dataset.from_list(rows, features=features)
    ds.save_to_disk(str(STYLE_DIR))
    print(f"  Saved {len(ds)} samples")
else:
    print("[skip] Style dataset already exists")

# 4/5 ControlNet pairs
if not (CONTROLNET_DIR.exists() and any(CONTROLNET_DIR.iterdir())):
    print("[4/5] Downloading ControlNet conditioning pairs...")
    ds = load_dataset("HighCWu/diffusion-db-2m-first-1k", split="train")
    ds.save_to_disk(str(CONTROLNET_DIR))
    print(f"  Saved {len(ds)} samples")
else:
    print("[skip] ControlNet dataset already exists")

# 5/5 Video clips
if not (VIDEO_DIR.exists() and any(VIDEO_DIR.iterdir())):
    print("[5/5] Downloading video clips (OpenVid-1M first shard)...")
    snapshot_download(
        repo_id="nkp37/OpenVid-1M", repo_type="dataset",
        local_dir=str(VIDEO_DIR),
        ignore_patterns=["*.parquet"],
        allow_patterns=["data/train-00000*", "README.md"],
    )
else:
    print("[skip] Video dataset already exists")

print("\n--- Verification ---")
for name, path in [("subject", SUBJECT_DIR), ("reg", REG_DIR), ("style", STYLE_DIR),
                    ("controlnet", CONTROLNET_DIR), ("video", VIDEO_DIR)]:
    ok = path.exists() and any(path.iterdir()) if path.exists() else False
    print(f"  [{'OK' if ok else 'MISSING'}] {name}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Phase 2 — Download Base Models

SDXL downloads automatically. FLUX.1-dev requires accepting the license first:  
https://huggingface.co/black-forest-labs/FLUX.1-dev  

Toggle which models to download below.

In [ ]:
DOWNLOAD_FLUX = False       # Set True if you have A100+ and accepted the license
DOWNLOAD_COGVIDEO = False   # Set True if you have A100+ (24GB+ VRAM)

# SDXL base
if not (SDXL_PATH.exists() and any(SDXL_PATH.iterdir())):
    print("Downloading SDXL 1.0 base...")
    snapshot_download(repo_id="stabilityai/stable-diffusion-xl-base-1.0", local_dir=str(SDXL_PATH))
else:
    print("[skip] SDXL base already exists")

# SDXL VAE fix
if not (SDXL_VAE_PATH.exists() and any(SDXL_VAE_PATH.iterdir())):
    print("Downloading SDXL VAE fp16 fix...")
    snapshot_download(repo_id="madebyollin/sdxl-vae-fp16-fix", local_dir=str(SDXL_VAE_PATH))
else:
    print("[skip] SDXL VAE fix already exists")

# FLUX.1-dev (optional)
if DOWNLOAD_FLUX:
    if not (FLUX_PATH.exists() and any(FLUX_PATH.iterdir())):
        print("Downloading FLUX.1-dev (this is large ~33GB)...")
        snapshot_download(repo_id="black-forest-labs/FLUX.1-dev", local_dir=str(FLUX_PATH), ignore_patterns=["*.gguf"])
    else:
        print("[skip] FLUX.1-dev already exists")

# CogVideoX (optional)
if DOWNLOAD_COGVIDEO:
    if not (COGVIDEO_PATH.exists() and any(COGVIDEO_PATH.iterdir())):
        print("Downloading CogVideoX-5b...")
        snapshot_download(repo_id="THUDM/CogVideoX-5b", local_dir=str(COGVIDEO_PATH))
    else:
        print("[skip] CogVideoX already exists")

print("\n--- Verification ---")
for name, path in [("SDXL-base", SDXL_PATH), ("SDXL-VAE", SDXL_VAE_PATH),
                    ("FLUX.1-dev", FLUX_PATH), ("CogVideoX", COGVIDEO_PATH)]:
    ok = path.exists() and any(path.iterdir()) if path.exists() else False
    print(f"  [{'OK' if ok else '---'}] {name}")

## Phase 3 — Preprocess (BLIP-2 Captioning + ControlNet Maps)

In [ ]:
import json
from PIL import Image
from datasets import load_from_disk

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- Auto-caption with BLIP-2 ---
from transformers import Blip2Processor, Blip2ForConditionalGeneration

print("Loading BLIP-2 (Salesforce/blip2-opt-2.7b)...")
blip_proc = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16, device_map="auto",
)

print(f"Loading subject dataset from {SUBJECT_DIR}...")
ds = load_from_disk(str(SUBJECT_DIR))

metadata = []
for i, sample in enumerate(ds):
    img = sample["image"].convert("RGB").resize((1024, 1024), Image.LANCZOS)
    inputs = blip_proc(images=img, return_tensors="pt").to("cuda", torch.float16)
    out = blip_model.generate(**inputs, max_new_tokens=50)
    caption = blip_proc.decode(out[0], skip_special_tokens=True)
    caption = f"a photo of {TRIGGER_TOKEN}, {caption}"

    img_path = PROCESSED_DIR / f"img_{i:04d}.jpg"
    img.save(str(img_path), quality=95)
    metadata.append({"file_name": str(img_path), "text": caption})
    print(f"  [{i+1}/{len(ds)}] {caption[:80]}...")

with open(PROCESSED_DIR / "metadata.jsonl", "w") as f:
    for m in metadata:
        f.write(json.dumps(m) + "\n")

print(f"\nSaved {len(metadata)} captioned images")

# Free BLIP-2 memory
del blip_model, blip_proc
torch.cuda.empty_cache()

In [ ]:
# --- Extract ControlNet conditioning maps ---
from controlnet_aux import OpenposeDetector, MidasDetector, CannyDetector

print("Extracting ControlNet conditioning maps...")
ds = load_from_disk(str(SUBJECT_DIR))

pose_det = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
depth_det = MidasDetector.from_pretrained("lllyasviel/ControlNet")
canny_det = CannyDetector()

for i, sample in enumerate(ds):
    img = sample["image"].convert("RGB").resize((1024, 1024))
    try: pose_det(img).save(str(PROCESSED_DIR / f"pose_{i:04d}.png"))
    except Exception as e: print(f"  [warn] Pose failed for {i}: {e}")
    try: depth_det(img).save(str(PROCESSED_DIR / f"depth_{i:04d}.png"))
    except Exception as e: print(f"  [warn] Depth failed for {i}: {e}")
    try: canny_det(img, low_threshold=100, high_threshold=200).save(str(PROCESSED_DIR / f"canny_{i:04d}.png"))
    except Exception as e: print(f"  [warn] Canny failed for {i}: {e}")
    print(f"  [{i+1}/{len(ds)}] done")

del pose_det, depth_det, canny_det
torch.cuda.empty_cache()
print("Preprocessing complete.")

## Phase 4 — Train Subject LoRA (DreamBooth)

Trains a subject-specific LoRA adapter with pivotal tuning.  
~1000 steps, takes ~15-30 min on T4.

In [ ]:
from torch.utils.data import DataLoader, Dataset as TorchDataset
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model

class DreamBoothDataset(TorchDataset):
    def __init__(self, data_dir):
        with open(data_dir / "metadata.jsonl") as f:
            self.samples = [json.loads(line) for line in f]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        return {"image": Image.open(s["file_name"]).convert("RGB"), "caption": s["text"]}

SUBJECT_LORA_PATH.mkdir(parents=True, exist_ok=True)
accelerator = Accelerator(mixed_precision="bf16", gradient_accumulation_steps=SUBJECT_GRAD_ACCUM)

# Load base model (FLUX if available, else SDXL)
if FLUX_PATH.exists() and any(FLUX_PATH.iterdir()):
    from diffusers import FluxPipeline
    pipe = FluxPipeline.from_pretrained(str(FLUX_PATH), torch_dtype=torch.bfloat16)
    transformer = pipe.transformer
    model_type = "flux"
else:
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(str(SDXL_PATH), torch_dtype=torch.bfloat16)
    transformer = pipe.unet
    model_type = "sdxl"
print(f"Loaded {model_type} model")

# LoRA setup
lora_cfg = LoraConfig(
    r=SUBJECT_LORA_RANK, lora_alpha=SUBJECT_LORA_ALPHA,
    target_modules=["to_q", "to_k", "to_v", "to_out.0", "ff.net.0.proj", "ff.net.2"],
    lora_dropout=0.05, bias="none",
)
transformer = get_peft_model(transformer, lora_cfg)
transformer.print_trainable_parameters()
transformer.gradient_checkpointing_enable()

# Pivotal tuning — add trigger token
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
num_added = tokenizer.add_tokens([TRIGGER_TOKEN])
text_encoder.resize_token_embeddings(len(tokenizer))
for name, param in text_encoder.named_parameters():
    param.requires_grad = (name == "text_model.embeddings.token_embedding.weight")
print(f"Added {num_added} trigger token(s)")

dataset = DreamBoothDataset(PROCESSED_DIR)
loader = DataLoader(dataset, batch_size=SUBJECT_BATCH_SIZE, shuffle=True)

trainable_params = list(transformer.parameters()) + list(text_encoder.get_input_embeddings().parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=SUBJECT_LR, weight_decay=1e-2)

transformer, text_encoder, optimizer, loader = accelerator.prepare(transformer, text_encoder, optimizer, loader)
vae = pipe.vae.to(accelerator.device, dtype=torch.bfloat16)
vae.requires_grad_(False)

print(f"\nTraining: {SUBJECT_STEPS} steps, batch={SUBJECT_BATCH_SIZE}, grad_accum={SUBJECT_GRAD_ACCUM}")

global_step = 0
while global_step < SUBJECT_STEPS:
    for batch in loader:
        with accelerator.accumulate(transformer):
            images = [img for img in batch["image"]]
            pixel_values = torch.stack([
                pipe.image_processor.preprocess(img) if hasattr(pipe, "image_processor")
                else torch.tensor([]) for img in images
            ]).to(accelerator.device, dtype=torch.bfloat16)

            latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor
            input_ids = tokenizer(batch["caption"], padding="max_length", truncation=True,
                                  max_length=77, return_tensors="pt").input_ids.to(accelerator.device)
            encoder_hidden_states = text_encoder(input_ids)[0]

            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, 1000, (latents.shape[0],), device=accelerator.device)
            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
            pred = transformer(noisy_latents, timestep=timesteps, encoder_hidden_states=encoder_hidden_states).sample

            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

        global_step += 1
        if global_step % 100 == 0:
            print(f"  Step {global_step}/{SUBJECT_STEPS} | loss: {loss.item():.4f}")
        if global_step >= SUBJECT_STEPS:
            break

unwrapped = accelerator.unwrap_model(transformer)
unwrapped.save_pretrained(str(SUBJECT_LORA_PATH))
print(f"\nSubject LoRA saved to {SUBJECT_LORA_PATH}")

del pipe, transformer, text_encoder, vae, optimizer
torch.cuda.empty_cache()

## Phase 5 — Train Style LoRA

Style LoRA on WikiArt. Lighter rank (32), 500 steps.  
~10-15 min on T4.

In [ ]:
class StyleDataset(TorchDataset):
    def __init__(self, data_dir):
        self.ds = load_from_disk(str(data_dir))
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        s = self.ds[idx]
        img = s["image"].convert("RGB").resize((1024, 1024), Image.LANCZOS)
        return {"image": img, "caption": f"{STYLE_TRIGGER}, {s.get('text', 'a painting')}"}

STYLE_LORA_PATH.mkdir(parents=True, exist_ok=True)
accelerator = Accelerator(mixed_precision="bf16", gradient_accumulation_steps=SUBJECT_GRAD_ACCUM)

if FLUX_PATH.exists() and any(FLUX_PATH.iterdir()):
    from diffusers import FluxPipeline
    pipe = FluxPipeline.from_pretrained(str(FLUX_PATH), torch_dtype=torch.bfloat16)
    transformer = pipe.transformer
else:
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(str(SDXL_PATH), torch_dtype=torch.bfloat16)
    transformer = pipe.unet

lora_cfg = LoraConfig(
    r=STYLE_LORA_RANK, lora_alpha=STYLE_LORA_ALPHA,
    target_modules=["to_q", "to_k", "to_v", "to_out.0", "ff.net.0.proj", "ff.net.2"],
    lora_dropout=0.0, bias="none",
)
transformer = get_peft_model(transformer, lora_cfg)
transformer.print_trainable_parameters()
transformer.gradient_checkpointing_enable()

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
text_encoder.requires_grad_(False)

dataset = StyleDataset(STYLE_DIR)
loader = DataLoader(dataset, batch_size=SUBJECT_BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(transformer.parameters(), lr=STYLE_LR, weight_decay=1e-2)

transformer, optimizer, loader = accelerator.prepare(transformer, optimizer, loader)
vae = pipe.vae.to(accelerator.device, dtype=torch.bfloat16)
vae.requires_grad_(False)
text_encoder = text_encoder.to(accelerator.device)

print(f"Training style LoRA: {STYLE_STEPS} steps")

global_step = 0
while global_step < STYLE_STEPS:
    for batch in loader:
        with accelerator.accumulate(transformer):
            images = [img for img in batch["image"]]
            pixel_values = torch.stack([
                pipe.image_processor.preprocess(img) if hasattr(pipe, "image_processor")
                else torch.tensor([]) for img in images
            ]).to(accelerator.device, dtype=torch.bfloat16)

            latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor
            input_ids = tokenizer(batch["caption"], padding="max_length", truncation=True,
                                  max_length=77, return_tensors="pt").input_ids.to(accelerator.device)
            encoder_hidden_states = text_encoder(input_ids)[0]

            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, 1000, (latents.shape[0],), device=accelerator.device)
            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
            pred = transformer(noisy_latents, timestep=timesteps, encoder_hidden_states=encoder_hidden_states).sample

            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

        global_step += 1
        if global_step % 100 == 0:
            print(f"  Step {global_step}/{STYLE_STEPS} | loss: {loss.item():.4f}")
        if global_step >= STYLE_STEPS:
            break

unwrapped = accelerator.unwrap_model(transformer)
unwrapped.save_pretrained(str(STYLE_LORA_PATH))
print(f"\nStyle LoRA saved to {STYLE_LORA_PATH}")

del pipe, transformer, text_encoder, vae, optimizer
torch.cuda.empty_cache()

## Phase 6 — ControlNet Inference

In [ ]:
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
from controlnet_aux import OpenposeDetector, CannyDetector, MidasDetector
from IPython.display import display

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading ControlNet models...")
controlnets = [
    ControlNetModel.from_pretrained("thibaud/controlnet-openpose-sdxl-1.0", torch_dtype=torch.float16),
    ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16),
    ControlNetModel.from_pretrained("diffusers/controlnet-depth-sdxl-1.0", torch_dtype=torch.float16),
]

vae = AutoencoderKL.from_pretrained(str(SDXL_VAE_PATH), torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    str(SDXL_PATH), controlnet=controlnets, vae=vae, torch_dtype=torch.float16,
)
pipe.enable_model_cpu_offload()

if SUBJECT_LORA_PATH.exists() and any(SUBJECT_LORA_PATH.iterdir()):
    pipe.load_lora_weights(str(SUBJECT_LORA_PATH), adapter_name="subject")
    pipe.set_adapters(["subject"], adapter_weights=[1.0])
    print("  Subject LoRA loaded")

source_path = PROCESSED_DIR / "img_0000.jpg"
source_img = Image.open(str(source_path)).convert("RGB") if source_path.exists() else Image.new("RGB", (1024, 1024), (128, 128, 128))

pose_det = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
canny_det = CannyDetector()
depth_det = MidasDetector.from_pretrained("lllyasviel/ControlNet")
cond_images = [pose_det(source_img), canny_det(source_img, 100, 200), depth_det(source_img)]

prompt = f"a photo of {TRIGGER_TOKEN} walking in Tokyo, {STYLE_TRIGGER}, cinematic lighting, 4k"
print(f"Generating: {prompt[:80]}...")

result = pipe(
    prompt=prompt,
    negative_prompt="ugly, blurry, low quality, deformed, disfigured",
    image=cond_images,
    controlnet_conditioning_scale=[0.8, 0.5, 0.5],
    num_inference_steps=DEFAULT_STEPS,
    guidance_scale=7.5,
    generator=torch.manual_seed(DEFAULT_SEED),
).images[0]

result.save(str(OUTPUT_DIR / "controlnet_result.png"))
display(result.resize((512, 512)))
print("Saved: outputs/controlnet_result.png")

del pipe, controlnets, vae
torch.cuda.empty_cache()

## Phase 7 — IP-Adapter Inference

In [ ]:
from diffusers import StableDiffusionXLPipeline
from diffusers.utils import load_image
from huggingface_hub import hf_hub_download

IP_ADAPTER_DIR = MODEL_DIR / "ip-adapter"

weight_path = IP_ADAPTER_DIR / "sdxl_models" / "ip-adapter_sdxl.bin"
if not weight_path.exists():
    print("Downloading IP-Adapter SDXL weights...")
    hf_hub_download(repo_id="h94/IP-Adapter", filename="sdxl_models/ip-adapter_sdxl.bin", local_dir=str(IP_ADAPTER_DIR))
    hf_hub_download(repo_id="h94/IP-Adapter", filename="models/image_encoder/config.json", local_dir=str(IP_ADAPTER_DIR))

pipe = StableDiffusionXLPipeline.from_pretrained(str(SDXL_PATH), torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
pipe.load_ip_adapter(str(IP_ADAPTER_DIR), subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
pipe.set_ip_adapter_scale(0.7)

if SUBJECT_LORA_PATH.exists() and any(SUBJECT_LORA_PATH.iterdir()):
    pipe.load_lora_weights(str(SUBJECT_LORA_PATH), adapter_name="subject")
    pipe.set_adapters(["subject"], adapter_weights=[1.0])

ref_path = PROCESSED_DIR / "img_0000.jpg"
ref_image = load_image(str(ref_path)) if ref_path.exists() else Image.new("RGB", (1024, 1024), (128, 128, 128))

prompt = f"a photo of {TRIGGER_TOKEN} in a futuristic city, cinematic"
print(f"Generating: {prompt}")

result = pipe(
    prompt=prompt, ip_adapter_image=ref_image,
    num_inference_steps=DEFAULT_STEPS, guidance_scale=6.0,
    generator=torch.manual_seed(DEFAULT_SEED),
).images[0]

result.save(str(OUTPUT_DIR / "ip_adapter_result.png"))
display(result.resize((512, 512)))
print("Saved: outputs/ip_adapter_result.png")

del pipe
torch.cuda.empty_cache()

## Phase 8 — Multi-LoRA Engine Test

In [ ]:
if FLUX_PATH.exists() and any(FLUX_PATH.iterdir()):
    from diffusers import FluxPipeline
    pipe = FluxPipeline.from_pretrained(str(FLUX_PATH), torch_dtype=torch.bfloat16)
else:
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(str(SDXL_PATH), torch_dtype=torch.float16)

pipe.enable_model_cpu_offload()

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xformers enabled")
except Exception:
    print("xformers not available, using SDPA")

adapters, weights = [], []
if SUBJECT_LORA_PATH.exists() and any(SUBJECT_LORA_PATH.iterdir()):
    pipe.load_lora_weights(str(SUBJECT_LORA_PATH), adapter_name="subject")
    adapters.append("subject"); weights.append(1.0)
    print("  Subject LoRA loaded")
if STYLE_LORA_PATH.exists() and any(STYLE_LORA_PATH.iterdir()):
    pipe.load_lora_weights(str(STYLE_LORA_PATH), adapter_name="style")
    adapters.append("style"); weights.append(0.6)
    print("  Style LoRA loaded")
if adapters:
    pipe.set_adapters(adapters, adapter_weights=weights)

prompt = f"a photo of {TRIGGER_TOKEN} on Mars, {STYLE_TRIGGER}"
print(f"Generating: {prompt}")

result = pipe(
    prompt=prompt, num_inference_steps=DEFAULT_STEPS,
    guidance_scale=DEFAULT_GUIDANCE, generator=torch.manual_seed(DEFAULT_SEED),
    width=1024, height=1024,
).images[0]

result.save(str(OUTPUT_DIR / "multi_lora_result.png"))
display(result.resize((512, 512)))
print("Saved: outputs/multi_lora_result.png")

del pipe
torch.cuda.empty_cache()

## Phase 9 — AnimateDiff Video Generation

In [ ]:
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_gif
from IPython.display import Image as IPImage, display

ANIMATEDIFF_ADAPTER_PATH = MODEL_DIR / "animatediff-motion-adapter"
ANIMATEDIFF_LORA_PATH = MODEL_DIR / "animatediff-motion-lora"

if not (ANIMATEDIFF_ADAPTER_PATH.exists() and any(ANIMATEDIFF_ADAPTER_PATH.iterdir())):
    print("Downloading AnimateDiff motion adapter...")
    snapshot_download(repo_id="guoyww/animatediff-motion-adapter-v1-5-2", local_dir=str(ANIMATEDIFF_ADAPTER_PATH))
if not (ANIMATEDIFF_LORA_PATH.exists() and any(ANIMATEDIFF_LORA_PATH.iterdir())):
    print("Downloading AnimateDiff motion LoRA...")
    snapshot_download(repo_id="guoyww/animatediff-motion-lora-v1-5-2", local_dir=str(ANIMATEDIFF_LORA_PATH))

adapter = MotionAdapter.from_pretrained(str(ANIMATEDIFF_ADAPTER_PATH))
pipe = AnimateDiffPipeline.from_pretrained(str(SDXL_PATH), motion_adapter=adapter, torch_dtype=torch.float16)
pipe.scheduler = DDIMScheduler.from_config(
    pipe.scheduler.config, beta_schedule="linear",
    clip_sample=False, timestep_spacing="linspace", steps_offset=1,
)
pipe.enable_model_cpu_offload()

if SUBJECT_LORA_PATH.exists() and any(SUBJECT_LORA_PATH.iterdir()):
    pipe.load_lora_weights(str(SUBJECT_LORA_PATH), adapter_name="subject")
    pipe.set_adapters(["subject"], adapter_weights=[1.0])

prompt = f"{TRIGGER_TOKEN} walking through a forest, cinematic, 4k"
print(f"Generating animation: {prompt}")

output = pipe(prompt=prompt, num_frames=16, guidance_scale=7.5,
              num_inference_steps=25, generator=torch.manual_seed(42))
frames = output.frames[0]

gif_path = str(OUTPUT_DIR / "animation.gif")
export_to_gif(frames, gif_path)
display(IPImage(filename=gif_path))
print(f"Saved: {gif_path}")

del pipe, adapter
torch.cuda.empty_cache()

## Phase 11 — Gradio Web UI

Launches a public Gradio link (works in Colab).

In [ ]:
import gradio as gr

engine = None

def get_engine():
    global engine
    if engine is None:
        if FLUX_PATH.exists() and any(FLUX_PATH.iterdir()):
            from diffusers import FluxPipeline
            pipe = FluxPipeline.from_pretrained(str(FLUX_PATH), torch_dtype=torch.bfloat16)
        else:
            from diffusers import StableDiffusionXLPipeline
            pipe = StableDiffusionXLPipeline.from_pretrained(str(SDXL_PATH), torch_dtype=torch.float16)
        pipe.enable_model_cpu_offload()
        if SUBJECT_LORA_PATH.exists() and any(SUBJECT_LORA_PATH.iterdir()):
            pipe.load_lora_weights(str(SUBJECT_LORA_PATH), adapter_name="subject")
        if STYLE_LORA_PATH.exists() and any(STYLE_LORA_PATH.iterdir()):
            pipe.load_lora_weights(str(STYLE_LORA_PATH), adapter_name="style")
        engine = pipe
    return engine

def generate_image(prompt, subject_weight, style_weight, steps, guidance, seed):
    pipe = get_engine()
    adapters, weights = [], []
    if subject_weight > 0:
        adapters.append("subject"); weights.append(subject_weight)
    if style_weight > 0:
        adapters.append("style"); weights.append(style_weight)
    if adapters:
        pipe.set_adapters(adapters, adapter_weights=weights)
    return pipe(
        prompt=prompt, num_inference_steps=int(steps),
        guidance_scale=guidance, generator=torch.manual_seed(int(seed)),
        width=1024, height=1024,
    ).images[0]

with gr.Blocks(title="DreamBooth LoRA Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# DreamBooth LoRA Studio\nPersonalized generation with LoRA adapters.")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Prompt", value=f"a photo of {TRIGGER_TOKEN} on Mars, {STYLE_TRIGGER}", lines=3)
            with gr.Row():
                subj_w = gr.Slider(0, 1.5, value=1.0, step=0.05, label="Subject LoRA")
                style_w = gr.Slider(0, 1.5, value=0.6, step=0.05, label="Style LoRA")
            with gr.Row():
                steps = gr.Slider(10, 50, value=DEFAULT_STEPS, step=1, label="Steps")
                guidance = gr.Slider(1.0, 15.0, value=DEFAULT_GUIDANCE, step=0.5, label="Guidance")
            seed = gr.Number(value=DEFAULT_SEED, label="Seed", precision=0)
            btn = gr.Button("Generate", variant="primary")
        with gr.Column():
            output = gr.Image(label="Output", type="pil")
    btn.click(generate_image, [prompt, subj_w, style_w, steps, guidance, seed], output)
    gr.Markdown(f"**Trigger token:** `{TRIGGER_TOKEN}` · **Style trigger:** `{STYLE_TRIGGER}`")

demo.launch(share=True)

## Copy Results to Google Drive

Run this before the runtime disconnects to save your outputs.

In [ ]:
import shutil

drive_outputs = os.path.join(DRIVE_ROOT, "outputs")
os.makedirs(drive_outputs, exist_ok=True)

local_outputs = str(OUTPUT_DIR)
copied = 0
for f in os.listdir(local_outputs):
    src = os.path.join(local_outputs, f)
    dst = os.path.join(drive_outputs, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        copied += 1

print(f"Copied {copied} file(s) to {drive_outputs}")